In [3]:
import os
import json
import shutil
import random
from pathlib import Path

# =========================================================
# 1. 원본 경로
# =========================================================
label_root = Path(r"C:\py_temp\new_proj\위험지역_사람포착\136.야간 사건사고 대응을 위한 버드아이뷰 IR열화상 데이터\01.데이터\1.Training\라벨링데이터")
image_root = Path(r"C:\py_temp\new_proj\위험지역_사람포착\136.야간 사건사고 대응을 위한 버드아이뷰 IR열화상 데이터\01.데이터\1.Training\원천데이터\TS_객체데이터이미지\객체데이터이미지")

# =========================================================
# 2. 결과 폴더
# =========================================================
save_root = Path(r"C:\py_temp\new_proj\위험지역_사람포착\dataset")

images_train_dir = save_root / "images" / "train"
images_val_dir   = save_root / "images" / "val"
labels_train_dir = save_root / "labels" / "train"
labels_val_dir   = save_root / "labels" / "val"

images_train_dir.mkdir(parents=True, exist_ok=True)
images_val_dir.mkdir(parents=True, exist_ok=True)
labels_train_dir.mkdir(parents=True, exist_ok=True)
labels_val_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# 3. 이미지 파일 전체 인덱싱
#    - 하위 폴더가 많으니 파일명 기준으로 빠르게 찾기 위해 딕셔너리 생성
# =========================================================
image_map = {}

for root, dirs, files in os.walk(image_root):
    for file in files:
        lower = file.lower()
        if lower.endswith(".jpg") or lower.endswith(".jpeg") or lower.endswith(".png"):
            # 파일명이 중복될 가능성은 낮다고 보고 파일명 기준 저장
            image_map[file] = str(Path(root) / file)

print(f"이미지 파일 수: {len(image_map)}")

# =========================================================
# 4. json 라벨 파일 전체 수집
# =========================================================
json_files = []

for root, dirs, files in os.walk(label_root):
    for file in files:
        if file.lower().endswith(".json"):
            json_files.append(str(Path(root) / file))

print(f"json 파일 수: {len(json_files)}")

# =========================================================
# 5. 사람 라벨이 있고, 이미지 매칭도 되는 샘플만 수집
# =========================================================
samples = []
missing_image_count = 0
no_person_count = 0
error_count = 0

for json_path in json_files:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if "image" not in data:
            error_count += 1
            continue

        if "file_name" not in data["image"]:
            error_count += 1
            continue

        file_name = data["image"]["file_name"]

        if file_name not in image_map:
            missing_image_count += 1
            continue

        if "annotation" not in data:
            no_person_count += 1
            continue

        person_lines = []

        for ann in data["annotation"]:
            if "property" not in ann:
                continue
            if "name" not in ann["property"]:
                continue

            class_name = ann["property"]["name"]

            # 사람만 사용
            if class_name != "사람":
                continue

            # entity_box가 있으면 그걸 우선 사용
            if "entity_box" in ann and len(ann["entity_box"]) == 4:
                cx, cy, w, h = ann["entity_box"]

                # YOLO 형식: class_id center_x center_y width height
                # class_id는 person 하나만 쓰므로 0
                line = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
                person_lines.append(line)

            # entity_box가 없으면 bndbox로 계산
            elif "bndbox" in ann:
                bbox = ann["bndbox"]

                if "size" not in data["image"]:
                    continue

                img_w = data["image"]["size"]["width"]
                img_h = data["image"]["size"]["height"]

                xmin = bbox["xmin"]
                ymin = bbox["ymin"]
                xmax = bbox["xmax"]
                ymax = bbox["ymax"]

                box_w = xmax - xmin
                box_h = ymax - ymin
                cx = xmin + box_w / 2
                cy = ymin + box_h / 2

                cx /= img_w
                cy /= img_h
                box_w /= img_w
                box_h /= img_h

                line = f"0 {cx:.6f} {cy:.6f} {box_w:.6f} {box_h:.6f}"
                person_lines.append(line)

        if len(person_lines) == 0:
            no_person_count += 1
            continue

        samples.append({
            "json_path": json_path,
            "image_path": image_map[file_name],
            "file_name": file_name,
            "label_lines": person_lines
        })

    except Exception as e:
        error_count += 1
        continue

print(f"최종 사용 가능 샘플 수: {len(samples)}")
print(f"이미지 매칭 실패 수: {missing_image_count}")
print(f"사람 라벨 없음 수: {no_person_count}")
print(f"읽기 오류 수: {error_count}")

# =========================================================
# 6. train / val 분리
# =========================================================
random.seed(42)
random.shuffle(samples)

train_ratio = 0.8
train_count = int(len(samples) * train_ratio)

train_samples = samples[:train_count]
val_samples = samples[train_count:]

print(f"train 샘플 수: {len(train_samples)}")
print(f"val 샘플 수: {len(val_samples)}")

# =========================================================
# 7. 파일 복사 + YOLO txt 저장
# =========================================================
# 파일명 충돌 방지를 위해 stem 기준 중복 체크
used_names = set()

for split_name, split_samples, img_dir, lbl_dir in [
    ("train", train_samples, images_train_dir, labels_train_dir),
    ("val", val_samples, images_val_dir, labels_val_dir)
]:
    for sample in split_samples:
        src_img = Path(sample["image_path"])
        stem = src_img.stem
        suffix = src_img.suffix.lower()

        save_name = stem
        idx = 1

        while save_name in used_names:
            save_name = f"{stem}_{idx}"
            idx += 1

        used_names.add(save_name)

        dst_img = img_dir / f"{save_name}{suffix}"
        dst_lbl = lbl_dir / f"{save_name}.txt"

        shutil.copy2(src_img, dst_img)

        with open(dst_lbl, "w", encoding="utf-8") as f:
            for line in sample["label_lines"]:
                f.write(line + "\n")

print("폴더 정리 완료")
print(f"결과 경로: {save_root}")

이미지 파일 수: 193015
json 파일 수: 928947
최종 사용 가능 샘플 수: 85690
이미지 매칭 실패 수: 771372
사람 라벨 없음 수: 71885
읽기 오류 수: 0
train 샘플 수: 68552
val 샘플 수: 17138
폴더 정리 완료
결과 경로: C:\py_temp\new_proj\위험지역_사람포착\dataset


In [1]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))
else:
    print("GPU를 못 잡고 있습니다.")

ModuleNotFoundError: No module named 'torch'